# Star-$R$ hypergraph study in $(\eta, \phi)$

We build **one hyperedge per active particle**: center $c$ connects to all neighbors with $\Delta R_{cj} < R$ in $(\eta,\phi)$. This broadens message-passing scope locally while keeping a fixed hyperedge count ($=\#$ active particles).

Before switching to **unnormalized message passing**, we study:
- support-size distributions vs $R$
- particle **membership degree** (how many hyperedges each particle belongs to)
- $\|TH\|/\|H\|$ operator scale ratios (normalized vs unnormalized)
- visual sanity checks on jet subsamples

Implementation: `cpen.utils.graph_radius`, `cpen.utils.graph_radius_viz`.

## How the construction works

For each active particle $c$:
1. Compute $\eta_c, \phi_c$ from the four-vector (see `eta_phi_from_four_vectors` in code).
2. Form support $S_c = \{c\} \cup \{j : \Delta R(c,j) < R\}$.
3. Incidence row $e_c$ has ones on $S_c$.

**Key properties:**
- $\#$ hyperedges $= \#$ active particles (padding slots excluded by $p_T$ mask).
- Hyperedge support sizes grow with $R$; at small $R$ most stars are isolated ($|S_c|=1$).
- A particle $j \neq c$ appears in star $e_c$ iff $j$ is within $\Delta R$ of $c$ — membership is **symmetric in $\Delta R$** but hyperedges are **directed stars** (each has a unique center).
- Particle $j$ belongs to all stars centered on neighbors within $R$ **plus** its own star $\Rightarrow$ node degree can exceed support size of any single hyperedge.

Contrast with momentum-space `build_star_hypergraph` (kNN in $(p_x,p_y,p_z)$): same star pattern but neighbor selection uses spatial distance, not $\Delta R$.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

REPO = Path("..").resolve()
sys.path.insert(0, str(REPO / "src"))

%load_ext autoreload
%autoreload 2

for mod in list(sys.modules):
    if mod == "cpen" or mod.startswith("cpen."):
        del sys.modules[mod]

from cpen.utils.graph_radius import (
    build_particle_radius_base_edges,
    build_star_radius_hypergraph,
    node_degree_histogram,
    particle_support_histogram,
)
from cpen.utils.graph_radius_viz import (
    plot_jet_subsample_panels,
    plot_star_hyperedge_panel,
    reference_particle_index,
)
from cpen.utils.toptagging import (
    N_PARTICLE_FEATURES,
    MAX_NUM_PARTICLES,
    raw_hdf5_path,
)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## Configuration

In [ ]:
DATA_ROOT = "/scratch/gpfs/BHANIN/jdezoort/datasets/toptagging"
SPLIT = "train"
NUM_PARTICLES = 100
N_TOP = 250
N_QCD = 250
SEED = 42

RADIUS_VALUES = np.round(np.linspace(0.05, 0.2, 6), 3)

VIZ_RADIUS = 0.08
N_VIZ_PER_CLASS = 4
VIZ_SEED = 42
OP_JET_SUBSAMPLE = 200
OP_WIDTH = 64

## Sample jets

In [ ]:
hdf5_path = raw_hdf5_path(DATA_ROOT, SPLIT)
assert hdf5_path.is_file(), f"Missing {hdf5_path}"

table = pd.read_hdf(hdf5_path, key="table")
total_pf = MAX_NUM_PARTICLES * N_PARTICLE_FEATURES
labels = table.iloc[:, -1].to_numpy(dtype=np.int64)

rng = np.random.default_rng(SEED)
top_pool = np.flatnonzero(labels == 1)
qcd_pool = np.flatnonzero(labels == 0)
top_idx = rng.choice(top_pool, size=N_TOP, replace=False)
qcd_idx = rng.choice(qcd_pool, size=N_QCD, replace=False)
sample_idx = np.concatenate([top_idx, qcd_idx])
sample_labels = np.concatenate([np.ones(N_TOP, dtype=np.int64), np.zeros(N_QCD, dtype=np.int64)])

pf_block = table.iloc[sample_idx, : total_pf].to_numpy(dtype=np.float32)
jets = pf_block.reshape(-1, MAX_NUM_PARTICLES, N_PARTICLE_FEATURES)[:, :NUM_PARTICLES]
jets_top = jets[sample_labels == 1]
jets_qcd = jets[sample_labels == 0]
print(f"sampled {len(jets)} jets: {N_TOP} top + {N_QCD} QCD")

## Scan $R$: support sizes, pairwise edges, node degree

Support histogram counts hyperedges by $|S_c|$. Pairwise edge count (no self-loops) is a reference for how many $\Delta R$ neighbor pairs exist. Node degree is the incidence column sum — how many stars each particle participates in.

In [ ]:
def scan_jet_star(jet: np.ndarray, radius: float) -> dict:
    hg = build_star_radius_hypergraph(jet, radius=radius)
    base_edges, _, _, _ = build_particle_radius_base_edges(
        jet, radius=radius, include_self_loops=False
    )
    active = hg.active_indices
    node_degrees = hg.incidence[:, active].sum(axis=0)
    return {
        "n_active": len(active),
        "n_hyperedges": hg.incidence.shape[0],
        "n_pairwise_edges": len(base_edges),
        "mean_support_size": float(hg.particle_support_sizes.mean()) if hg.incidence.shape[0] else 0.0,
        "mean_node_degree": float(node_degrees.mean()) if active.size else 0.0,
        "support_hist": particle_support_histogram(hg),
        "degree_hist": node_degree_histogram(hg),
    }


def aggregate_class_star(jets_cls: np.ndarray, radius: float) -> dict:
    rows = [scan_jet_star(jet, radius) for jet in tqdm(jets_cls, leave=False)]
    support_hist_total: dict[int, int] = {}
    degree_hist_total: dict[int, int] = {}
    for row in rows:
        for k, v in row["support_hist"].items():
            support_hist_total[k] = support_hist_total.get(k, 0) + v
        for k, v in row["degree_hist"].items():
            degree_hist_total[k] = degree_hist_total.get(k, 0) + v
    return {
        "radius": radius,
        "mean_active": float(np.mean([r["n_active"] for r in rows])),
        "mean_hyperedges": float(np.mean([r["n_hyperedges"] for r in rows])),
        "mean_pairwise_edges": float(np.mean([r["n_pairwise_edges"] for r in rows])),
        "mean_support_size": float(np.mean([r["mean_support_size"] for r in rows])),
        "mean_node_degree": float(np.mean([r["mean_node_degree"] for r in rows])),
        "support_hist_total": support_hist_total,
        "degree_hist_total": degree_hist_total,
    }


records_top = []
records_qcd = []
for R in tqdm(RADIUS_VALUES, desc="star-R scan"):
    records_top.append({**aggregate_class_star(jets_top, R), "class": "top"})
    records_qcd.append({**aggregate_class_star(jets_qcd, R), "class": "qcd"})

df_top = pd.DataFrame([{k: v for k, v in r.items() if not k.endswith("_hist_total")} for r in records_top])
df_qcd = pd.DataFrame([{k: v for k, v in r.items() if not k.endswith("_hist_total")} for r in records_qcd])
df_top.head()

## Plots: mean support size & node degree vs $R$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col, ylabel in zip(
    axes,
    ["mean_support_size", "mean_node_degree"],
    ["mean hyperedge support $|S_c|$", "mean node degree (column sum of $S$)"],
):
    ax.plot(df_top["radius"], df_top[col], marker="o", label="top")
    ax.plot(df_qcd["radius"], df_qcd[col], marker="s", label="QCD")
    ax.set_xlabel(r"radius $R$")
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)
fig.suptitle("Star-$R$ scaling", y=1.02)
fig.tight_layout()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_top["radius"], df_top["mean_pairwise_edges"], marker="o", label="top")
ax.plot(df_qcd["radius"], df_qcd["mean_pairwise_edges"], marker="s", label="QCD")
ax.set_xlabel(r"radius $R$")
ax.set_ylabel("mean pairwise $\Delta R$ edges (no self-loops)")
ax.set_title("Reference: undirected radius graph edge count")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()

## Stacked support-size histograms

In [ ]:
def stacked_support_plot(records, title: str, max_m: int = 20):
    radii = [r["radius"] for r in records]
    xs = np.arange(len(radii))
    bottom = np.zeros(len(radii))
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for m in range(1, max_m + 1):
        counts = np.array([r["support_hist_total"].get(m, 0) for r in records], dtype=float)
        if counts.sum() == 0:
            continue
        ax.bar(xs, counts, bottom=bottom, label=f"|S|={m}")
        bottom += counts
    ax.set_xticks(xs)
    ax.set_xticklabels([f"{r:.2f}" for r in radii], rotation=45, ha="right")
    ax.set_xlabel(r"radius $R$")
    ax.set_ylabel("total hyperedge count (summed over jets)")
    ax.set_title(title)
    ax.legend(title="support size", ncol=5, fontsize=9)
    ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    return fig

stacked_support_plot(records_top, "Top jets — star support sizes")
stacked_support_plot(records_qcd, "QCD jets — star support sizes")

## Visualize star hyperedges on jet subsamples

Left panel: only the star **centered on** the reference (highest-$E$) particle.
Right panel: all stars **incident on** that particle (it appears as a member), colored by $|S|$.

In [ ]:
plot_jet_subsample_panels(
    jets,
    sample_labels,
    radius=VIZ_RADIUS,
    n_per_class=N_VIZ_PER_CLASS,
    seed=VIZ_SEED,
)

In [ ]:
example_jet = jets_top[0]
ref = reference_particle_index(example_jet)
fig, ref, hg = plot_star_hyperedge_panel(example_jet, radius=VIZ_RADIUS, particle_idx=ref)
print(f"reference particle {ref}, E={example_jet[ref, 0]:.2f}")
print(f"active={len(hg.active_indices)}, hyperedges={hg.incidence.shape[0]}")
print(f"support histogram: {particle_support_histogram(hg)}")
print(f"node degree histogram: {node_degree_histogram(hg)}")

## Example: incidence matrix heatmap

In [ ]:
EXAMPLE_R = 0.08
hg = build_star_radius_hypergraph(example_jet, radius=EXAMPLE_R)
active = hg.active_indices

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(hg.incidence[:, active], aspect="auto", interpolation="nearest", cmap="Blues")
ax.set_xlabel("active particle index")
ax.set_ylabel("hyperedge index (star center order)")
ax.set_title(f"Incidence $S$ — star-$R$, R={EXAMPLE_R}")
plt.colorbar(im, ax=ax, fraction=0.02)

## Operator scale: normalized vs unnormalized

Report $\|TH\|/\|H\|$ for $T_{12}=D_E^{-1}S$, $T_{21}=D_X^{-1}S^\top$, $T_{22}=D_E^{-1}SD_X^{-1}S^\top$ and their unnormalized counterparts. The unnormalized ratios are the compensation targets if we drop degree weights.

In [ ]:
import torch
from cpen.utils.graphs import (
    apply_t12,
    apply_t21,
    apply_t22,
    apply_t12_unnorm,
    apply_t21_unnorm,
    apply_t22_unnorm,
    incidence_degrees,
)


def operator_scale_stats(jets_cls: np.ndarray, radius: float) -> dict[str, float]:
    ratios = {f"{kind}_{op}": [] for kind in ("norm", "unnorm") for op in ("t12", "t21", "t22")}
    for jet in jets_cls[:OP_JET_SUBSAMPLE]:
        hg = build_star_radius_hypergraph(jet, radius=radius)
        S = torch.from_numpy(hg.incidence).unsqueeze(0)
        if S.size(1) == 0:
            continue
        h_x = torch.randn(1, jet.shape[0], OP_WIDTH)
        h_x = h_x / h_x.norm(dim=-1, keepdim=True).clamp_min(1e-12)
        d_x_inv, d_e_inv = incidence_degrees(S)

        h_e_norm = apply_t12(h_x, S, d_e_inv)
        h_e_unnorm = apply_t12_unnorm(h_x, S)
        t21_norm = apply_t21(h_e_norm, S, d_x_inv)
        t21_unnorm = apply_t21_unnorm(h_e_unnorm, S)
        t22_norm = apply_t22(h_e_norm, S, d_x_inv, d_e_inv)
        t22_unnorm = apply_t22_unnorm(h_e_unnorm, S)

        ratios["norm_t12"].append((h_e_norm.norm() / h_x.norm()).item())
        ratios["norm_t21"].append((t21_norm.norm() / h_e_norm.norm()).item())
        ratios["norm_t22"].append((t22_norm.norm() / h_e_norm.norm()).item())
        ratios["unnorm_t12"].append((h_e_unnorm.norm() / h_x.norm()).item())
        ratios["unnorm_t21"].append((t21_unnorm.norm() / h_e_unnorm.norm()).item())
        ratios["unnorm_t22"].append((t22_unnorm.norm() / h_e_unnorm.norm()).item())

    return {k: float(np.mean(v)) for k, v in ratios.items()}


def scan_operator_ratios(jets_cls: np.ndarray, class_name: str) -> pd.DataFrame:
    rows = []
    for R in tqdm(RADIUS_VALUES, desc=f"operators {class_name}"):
        rows.append({"radius": R, "class": class_name, **operator_scale_stats(jets_cls, R)})
    return pd.DataFrame(rows)


op_top = scan_operator_ratios(jets_top, "top")
op_qcd = scan_operator_ratios(jets_qcd, "QCD")
op_df = pd.concat([op_top, op_qcd], ignore_index=True)
op_df.head()

In [ ]:
def plot_operator_ratio_scan(op_df: pd.DataFrame, operator: str):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, cls in zip(axes, ["top", "QCD"]):
        sub = op_df[op_df["class"] == cls]
        for normalized, alpha, marker in [(True, 0.95, "o"), (False, 0.55, None)]:
            col = f"{'norm' if normalized else 'unnorm'}_{operator}"
            label = "normalized" if normalized else "unnormalized"
            ax.plot(
                sub["radius"], sub[col], alpha=alpha, marker=marker, label=label
            )
        ax.set_xlabel(r"radius $R$")
        ax.set_title(cls)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=9)
    axes[0].set_ylabel(r"$\|TH\| / \|H\|$")
    fig.suptitle(f"Star-$R$ operator scale — {operator}", y=1.02)
    fig.tight_layout()
    return fig

for op in ("t12", "t21", "t22"):
    plot_operator_ratio_scan(op_df, op)

## Save summaries

In [ ]:
out_dir = REPO / "notebooks" / "outputs"
out_dir.mkdir(exist_ok=True)
df_top.to_csv(out_dir / "star_radius_scan_top_summary.csv", index=False)
df_qcd.to_csv(out_dir / "star_radius_scan_qcd_summary.csv", index=False)
op_df.to_csv(out_dir / "star_operator_ratio_scan.csv", index=False)
print(f"wrote {out_dir}")